In [ ]:
from geonate import common, raster, plot
import folium
import numpy as np
import rasterio

import folium
import rasterio
import numpy as np
from folium.raster_layers import ImageOverlay
from rasterio.plot import reshape_as_image
import pyproj

In [ ]:
geotiff_path = './Sample_data/landsat_multi/Scene/landsat_img_00.tif'

In [ ]:
def plot_basemap(image, basemap='OSM'):
    import folium
    import rasterio
    

In [ ]:
from folium import Map



tile = 'light'

if tile.lower() == 'openstreetmap' or tile.lower() == 'osm' or tile.lower() == 'open street map':
    tile_name = 'OpenStreetMap'
elif tile.lower() == 'cartodbpositron' or tile.lower() == 'cartodb positron' or tile.lower() == 'light' :
    tile_name = 'Cartodb Positron'
elif tile.lower() == 'cartodbdarkmatter' or tile.lower() == 'cartodb dark matter' or tile.lower() == 'dark':
    tile_name = 'Cartodb dark_matter'
elif tile.lower() == 



m = Map(location=(9.5, 105),  zoom_start= 10, tiles= tile_name)

m

In [ ]:
import folium
import rasterio
import numpy as np
from folium.raster_layers import ImageOverlay
from rasterio.plot import reshape_as_image
import pyproj


# Create a projection object for UTM (EPSG:32633 for UTM zone 33N as an example)
# You need to adjust this depending on your UTM zone
utm_proj = pyproj.CRS("EPSG:32648")  # Change EPSG code to your UTM zone (e.g., 32633 for UTM 33N)
latlon_proj = pyproj.CRS("EPSG:4326")  # WGS84 (Lat/Lon)

# Create a transformer for coordinate transformation
transformer = pyproj.Transformer.from_crs(utm_proj, latlon_proj, always_xy=True)

# Read the GeoTIFF file using rasterio
with rasterio.open(geotiff_path) as src:
    # Get the image data as numpy array
    image_data = reshape_as_image(src.read([4,3,2]))  # For RGB channels (adjust based on your bands)
    
    # Get the affine transformation for georeferencing
    bounds = src.bounds  # (left, bottom, right, top)
    
    # Convert the UTM bounds to Lat/Lon
    bottom_left = transformer.transform(bounds[0], bounds[1])  # (lon, lat)
    top_right = transformer.transform(bounds[2], bounds[3])    # (lon, lat)
    
    # Coordinates in Latitude/Longitude for the bounds
    lat_lon_bounds = [[bottom_left[1], bottom_left[0]], [top_right[1], top_right[0]]]

# Set up the center of the map (latitude, longitude) - use the center of your bounding box
latitude = (bottom_left[1] + top_right[1]) / 2
longitude = (bottom_left[0] + top_right[0]) / 2

# Create a Folium map centered at the GeoTIFF's location (in Lat/Lon)


m = folium.Map(location=[latitude, longitude], zoom_start=5)
folium.TileLayer(
    tiles='https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png',
    attr='&copy; Stadia Maps',
    name='Stadia.AlidadeSatellite'
).add_to(m)
'''
folium.TileLayer(
    tiles='https://basemap.nationalmap.gov/arcgis/rest/services/USGSImageryOnly/MapServer/tile/{z}/{y}/{x}',
    attr='&copy; Stadia Maps',
    name='Stadia.AlidadeSatellite'
).add_to(m)
'''
# Overlay the GeoTIFF image on the map
image_overlay = ImageOverlay(
    image=image_data,
    bounds=lat_lon_bounds,  # Use Lat/Lon bounds
    opacity=1
)

# Add the image overlay to the map
image_overlay.add_to(m)

# Save the map as an HTML file
#m.save("map_with_geotiff_and_basemap_utm_to_latlon.html")

# Display the map (if you're in a Jupyter notebook)
m


In [ ]:
from skimage import exposure

# Perform histogram equalization
image_eq = exposure.equalize_hist(image_data)  # This returns a floating point image with values between 0 and 1
image_eq = (image_eq * 255).astype(np.uint8)  # Convert back to 8-bit image for display

# Overlay the histogram equalized image on the map
image_overlay_eq = ImageOverlay(
    image=image_eq,
    bounds=lat_lon_bounds,  # Use Lat/Lon bounds
    opacity=0.7
)

# Add the equalized image overlay to the map
image_overlay_eq.add_to(m)


# Display the map (if you're in a Jupyter notebook)
m


In [ ]:
import folium
import rasterio
import numpy as np
from folium.raster_layers import ImageOverlay
from rasterio.plot import reshape_as_image
import pyproj
import matplotlib.pyplot as plt


# Create a projection object for UTM (change the EPSG code based on your UTM zone)
utm_proj = pyproj.CRS("EPSG:32648")  # Replace with your UTM zone (e.g., 32633 for UTM 33N)
latlon_proj = pyproj.CRS("EPSG:4326")  # WGS84 (Lat/Lon)

# Create a transformer for coordinate transformation
transformer = pyproj.Transformer.from_crs(utm_proj, latlon_proj, always_xy=True)

# Read the GeoTIFF file using rasterio
with rasterio.open(geotiff_path) as src:
    # Get the image data as a NumPy array (for RGB bands)
    image_data = reshape_as_image(src.read([4,3,2]))  # Adjust if you're using a different number of bands
    
    # Perform linear stretching on the image
    # Stretch the pixel values from the min to the max value (0 to 255 for RGB)
    image_stretched = np.clip((image_data - image_data.min()) / (image_data.max() - image_data.min()) * 255, 0, 255).astype(np.uint8)
    
    # Get the affine transformation for georeferencing
    bounds = src.bounds  # (left, bottom, right, top)
    
    # Convert the UTM bounds to Lat/Lon
    bottom_left = transformer.transform(bounds[0], bounds[1])  # (lon, lat)
    top_right = transformer.transform(bounds[2], bounds[3])    # (lon, lat)
    
    # Coordinates in Latitude/Longitude for the bounds
    lat_lon_bounds = [[bottom_left[1], bottom_left[0]], [top_right[1], top_right[0]]]

# Set up the center of the map (latitude, longitude) - use the center of your bounding box
latitude = (bottom_left[1] + top_right[1]) / 2
longitude = (bottom_left[0] + top_right[0]) / 2

# Create a Folium map centered at the GeoTIFF's location (in Lat/Lon)
m = folium.Map(location=[latitude, longitude], zoom_start=12, tiles="OpenStreetMap")

# Overlay the stretched image on the map
image_overlay = ImageOverlay(
    image=image_stretched,
    bounds=lat_lon_bounds,  # Use Lat/Lon bounds
    opacity=1
)

# Add the image overlay to the map
image_overlay.add_to(m)

# Save the map as an HTML file
m.save("map_with_stretched_geotiff.html")

# Display the map (if you're in a Jupyter notebook)
m
